# Pika H3 Flex · Colab and Google Drive benchmark

Run this from VS Code with the Colab extension. Create an **A100 → High-RAM → Latest** server and choose its Python kernel. The test uses the [ComfyUI Pika H3 Flex text-to-video workflow](https://docs.comfy.org/tutorials/video/pika/pika-h3-flex) and about 42 GB of checkpoint files. The weights are covered by the [MiniMax H3 Community License](https://huggingface.co/MiniMaxAI/MiniMax-H3/blob/main/LICENSE).

On the first server run **Setup** and **Save and render**. Wait for Drive to flush. Remove the server with **Colab → Remove Server**, reopen the notebook in a new VS Code window, and create another A100 server. Run **Setup** and **Cold reload** on that server. The cold timings exclude authorization and ComfyUI installation.

### Setup · Both VMs

In [ ]:
import json
import os
import re
import shutil
import socket
import subprocess
import sys
import time
from pathlib import Path

import requests
from google.colab import drive

repo_id = "Comfy-Org/MiniMax-H3"
revision = "4b53a35191437de05370da548e59aeabe2a5a09a"
comfy_version = "v0.45.0"
files = {
    "diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors": 20_970_379_616,
    "loras/pika_h3_flex_fl2va_v1_bf16.safetensors": 1_956_172_304,
    "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors": 15_687_142_551,
    "vae/minimax_h3_video_vae_int8_convrot.safetensors": 2_811_065_184,
    "vae/minimax_h3_audio_vae_fp32.safetensors": 605_254_808,
}
drive_folder = Path("/content/drive/MyDrive/colab-models/Pika-H3-Flex")
drive_models = drive_folder / "models"
work = Path("/content/pika-h3-flex-benchmark")
comfy = work / "ComfyUI"
local_models = comfy / "models"
runtime_marker = work / "baseline.marker"
vm_id = f"{socket.gethostname()}:{Path('/proc/sys/kernel/random/boot_id').read_text().strip()}"
gpu = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True
).strip()
print(f"GPU: {gpu} | VM: {socket.gethostname()}", flush=True)

drive.mount("/content/drive", timeout_ms=300_000)
if not drive_folder.parent.parent.is_dir():
    raise RuntimeError("Google Drive did not mount.")
work.mkdir(parents=True, exist_ok=True)

def complete(path, expected_bytes):
    return path.is_file() and path.stat().st_size == expected_bytes

def copy_model(source, destination, expected_bytes, allow_reuse=True):
    if allow_reuse and complete(destination, expected_bytes):
        return
    if not complete(source, expected_bytes):
        raise IOError(f"Source model is missing or incomplete: {source}")
    destination.parent.mkdir(parents=True, exist_ok=True)
    partial = destination.with_name(destination.name + ".partial")
    # Publish the finished file only after verifying its byte count.
    with source.open("rb") as reader, partial.open("wb") as writer:
        for index, block in enumerate(iter(lambda: reader.read(16 * 2**20), b""), 1):
            writer.write(block)
            if index % 256 == 0:
                print(f"  {index / 64:.0f} GiB copied", flush=True)
    if not complete(partial, expected_bytes):
        raise IOError(f"Incomplete model copy: {partial}")
    partial.replace(destination)

def install_comfy():
    if not (comfy / "main.py").is_file():
        subprocess.run([
            "git", "clone", "--quiet", "--depth", "1", "--branch", comfy_version,
            "https://github.com/Comfy-Org/ComfyUI.git", str(comfy)
        ], check=True)
    print(f"Preparing ComfyUI {comfy_version}", flush=True)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "-r", str(comfy / "requirements.txt")
    ], check=True)

def workflow(seed, phase):
    def node(kind, **inputs):
        return {"class_type": kind, "inputs": inputs}

    prompt = (
        "integrated_multimodal_description: At sunrise a little paper sailboat drifts across "
        "a still rain puddle. The camera tracks smoothly beside it at water level. Warm light, "
        "soft reflections, natural motion.\n\n"
        "overall_soundscape: a quiet trickle of water and soft morning birds.\n\n"
        "non_diegetic_music: none."
    )
    denoiser, lora, encoder, video_vae, audio_vae = (Path(name).name for name in files)
    return {
        "1": node("UNETLoader", unet_name=denoiser, weight_dtype="default"),
        "2": node("LoraLoaderModelOnly", model=["1",0], lora_name=lora, strength_model=1.0),
        "3": node("CLIPLoader", clip_name=encoder, type="minimax", device="default"),
        "4": node("VAELoader", vae_name=video_vae),
        "5": node("VAELoader", vae_name=audio_vae),
        "6": node("MiniMaxH3ImageToVideo", clip=["3",0], vae=["4",0], prompt=prompt,
                  width=768, height=448, length=56),
        "7": node("BasicGuider", model=["2",0], conditioning=["6",0]),
        "8": node("RandomNoise", noise_seed=seed),
        "9": node("KSamplerSelect", sampler_name="lcm"),
        "10": node("ManualSigmas", sigmas="1.0, 0.972973, 0.923077, 0.8, 0.571429, 0.333333, 0.0"),
        "11": node("SamplerCustomAdvanced", noise=["8",0], guider=["7",0], sampler=["9",0],
                   sigmas=["10",0], latent_image=["6",1]),
        "12": node("VAEDecode", samples=["11",0], vae=["4",0]),
        "13": node("VAEDecodeAudio", samples=["11",0], vae=["5",0]),
        "14": node("CreateVideo", images=["12",0], audio=["13",0], fps=24),
        "15": node("SaveVideo", video=["14",0], filename_prefix=f"pika_h3/{phase}",
                   format="mp4", codec="h264"),
    }

def render_video(phase):
    output_dir = work / "output"
    log_path = work / f"{phase}-comfy.log"
    base_url = "http://127.0.0.1:8188"
    with log_path.open("w") as log:
        server = subprocess.Popen([
            sys.executable, "-u", "main.py", "--listen", "127.0.0.1", "--port", "8188",
            "--offline", "--disable-all-custom-nodes", "--disable-auto-launch",
            "--output-directory", str(output_dir)
        ], cwd=comfy, stdout=log, stderr=subprocess.STDOUT)
        try:
            start_up = time.perf_counter()
            while True:
                if server.poll() is not None or time.perf_counter() - start_up > 240:
                    raise RuntimeError("ComfyUI failed to start.")
                try:
                    if requests.get(f"{base_url}/system_stats", timeout=2).ok:
                        break
                except requests.RequestException:
                    pass
                time.sleep(1)
            startup_seconds = time.perf_counter() - start_up
            print(f"ComfyUI started in {startup_seconds:.1f} s. Rendering {phase}.", flush=True)
            started = time.perf_counter()
            response = requests.post(
                f"{base_url}/prompt", json={"prompt": workflow(101, phase)}, timeout=30
            )
            if not response.ok:
                raise RuntimeError(f"ComfyUI rejected the workflow: {response.text}")
            prompt_id = response.json()["prompt_id"]
            next_report = 45
            while True:
                elapsed = time.perf_counter() - started
                if server.poll() is not None or elapsed > 1_500:
                    raise RuntimeError("ComfyUI exited or the render exceeded 25 minutes.")
                job = requests.get(f"{base_url}/history/{prompt_id}", timeout=10).json().get(prompt_id)
                if job:
                    if job["status"]["status_str"] != "success":
                        raise RuntimeError(f"ComfyUI render failed: {job['status']}")
                    break
                if elapsed >= next_report:
                    print(f"Render still running at {elapsed:.0f} s.", flush=True)
                    next_report += 45
                time.sleep(1)
            render_seconds = time.perf_counter() - started
            videos = sorted(output_dir.glob(f"pika_h3/{phase}_*.mp4"))
            if not videos or not videos[-1].stat().st_size:
                raise RuntimeError("ComfyUI finished without a video file.")
            target = drive_folder / f"{phase}.mp4"
            shutil.copyfile(videos[-1], target)
            print(f"First clip in {render_seconds:.1f} s. Saved {target}", flush=True)
            return {"comfy_start_seconds": startup_seconds,
                    "first_clip_seconds": render_seconds, "video_filename": target.name}
        except BaseException:
            print(log_path.read_text(errors="replace")[-6000:], flush=True)
            raise
        finally:
            if server.poll() is None:
                server.terminate()
                try:
                    server.wait(timeout=15)
                except subprocess.TimeoutExpired:
                    server.kill()
                    server.wait()

print(f"Drive destination: {drive_folder}", flush=True)

### Save and render · First A100

In [ ]:
from huggingface_hub import hf_hub_download

drive_folder.mkdir(parents=True, exist_ok=True)
pending_bytes = sum(size for name, size in files.items() if not complete(drive_models / name, size))
available = shutil.disk_usage(drive_folder).free
if available < pending_bytes:
    raise RuntimeError(f"Need {pending_bytes / 1e9:.1f} GB free in Google Drive.")
install_comfy()

started = time.perf_counter()
for name, size in files.items():
    target = drive_models / name
    if complete(target, size):
        print(f"Reusing Drive copy of {Path(name).name}", flush=True)
        copy_model(target, local_models / name, size)
        continue
    print(f"Downloading {Path(name).name} ({size / 1e9:.2f} GB)", flush=True)
    downloaded = Path(hf_hub_download(
        repo_id, name, revision=revision, local_dir=local_models, token=False
    ))
    print(f"Saving {Path(name).name} to Drive", flush=True)
    copy_model(downloaded, target, size)
persist_seconds = time.perf_counter() - started
runtime_marker.write_text(vm_id)
baseline = {"revision": revision, "vm_id": vm_id, "gpu": gpu,
            "model_bytes": sum(files.values()), "save_seconds": persist_seconds}
(drive_folder / "baseline.json").write_text(json.dumps(baseline, indent=2) + "\n")
print(f"Checkpoint saved in {persist_seconds:.1f} s. Rendering baseline.", flush=True)
baseline.update(render_video("baseline"))
(drive_folder / "baseline.json").write_text(json.dumps(baseline, indent=2) + "\n")
drive.flush_and_unmount()
print("Baseline complete. Drive flush finished.", flush=True)

### Cold reload · Fresh A100

In [ ]:
baseline = json.loads((drive_folder / "baseline.json").read_text())
if runtime_marker.exists() or vm_id == baseline["vm_id"]:
    raise RuntimeError("Provision a fresh VM. The baseline runtime is still in use.")
if baseline["revision"] != revision:
    raise RuntimeError("The Drive baseline used a different model revision.")
if any((local_models / name).exists() for name in files):
    raise RuntimeError("The runtime already contains model files. Use a fresh VM for cold timing.")
if shutil.disk_usage(work).free < sum(files.values()):
    raise RuntimeError("Insufficient local disk space for a cold reload.")

install_comfy()
os.environ["HF_HUB_OFFLINE"] = "1"
started = time.perf_counter()
per_file = {}
for name, size in files.items():
    print(f"Reloading {Path(name).name} from Drive", flush=True)
    file_started = time.perf_counter()
    copy_model(drive_models / name, local_models / name, size, allow_reuse=False)
    per_file[name] = round(time.perf_counter() - file_started, 2)
copy_seconds = time.perf_counter() - started
print(f"Cold Drive-to-disk reload: {copy_seconds:.1f} s ({sum(files.values()) / 1e9:.2f} GB)", flush=True)
result = render_video("after-restart")
result.update({
    "gpu": gpu, "baseline_vm_id": baseline["vm_id"], "reload_vm_id": vm_id,
    "model_bytes": sum(files.values()), "drive_to_disk_seconds": copy_seconds,
    "per_file_drive_seconds": per_file,
    "cold_total_seconds": copy_seconds + result["comfy_start_seconds"] + result["first_clip_seconds"],
})
(drive_folder / "reload-timing.json").write_text(json.dumps(result, indent=2) + "\n")
print(f"Drive-to-first-clip: {result['cold_total_seconds']:.1f} s", flush=True)
drive.flush_and_unmount()
print("Fresh-VM benchmark complete. Drive flush finished.", flush=True)